# Exploration of Water Quality Station (WQS) Datasets

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the downloaded result records and selected station metadata.
PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "data/raw/wqp/results"
DOWNLOAD_DIR = RESULTS_DIR / "pilot_download"

print("Results directory exists:", DOWNLOAD_DIR.exists())
print("Notebook project root:", PROJECT_ROOT)

Results directory exists: True
Notebook project root: /courses/DS5500.202710/teams/team4


In [2]:
manifest = pd.read_csv(DOWNLOAD_DIR / "download_manifest.csv")

display(manifest)

print("\nBatch status:")
display(manifest["status"].value_counts().to_frame("batches"))

if "rows" in manifest.columns:
    print("Total reported rows:", manifest["rows"].sum())

,batch,station_count,status,file,rows,error
0,1,5,success,results_batch_001.csv,2917,NaN
1,2,5,success,results_batch_002.csv,2007,NaN
2,3,5,success,results_batch_003.csv,372,NaN
3,4,5,success,results_batch_004.csv,335,NaN
4,5,5,success,results_batch_005.csv,977,NaN
5,6,1,zero_rows,results_batch_006.csv,0,NaN



Batch status:


,batches
status,
success,5
zero_rows,1


Total reported rows: 6608


## Load and combine files:

In [3]:
# Find all batch CSVs, excluding the manifest.
batch_files = sorted(DOWNLOAD_DIR.glob("results_batch_*.csv"))

print(f"Found {len(batch_files)} batch files")

# Read each batch and retain its filename for troubleshooting.
frames = []

for file in batch_files:
    try:
        df = pd.read_csv(file, low_memory=False)
        df["source_file"] = file.name
        frames.append(df)
        print(f"{file.name}: {len(df):,} rows")
    except Exception as exc:
        print(f"Could not read {file.name}: {exc}")

if not frames:
    raise ValueError("No result batches could be read.")

results = pd.concat(frames, ignore_index=True)

print(f"\nCombined shape: {results.shape}")
print(f"Unique columns: {results.shape[1]}")

Found 6 batch files
results_batch_001.csv: 2,917 rows
results_batch_002.csv: 2,007 rows
results_batch_003.csv: 372 rows
results_batch_004.csv: 335 rows
results_batch_005.csv: 977 rows
results_batch_006.csv: 0 rows

Combined shape: (6608, 64)
Unique columns: 64


In [4]:
results.head()

,OrganizationIdentifier,OrganizationFormalName,ActivityIdentifier,ActivityTypeCode,ActivityMediaName,ActivityMediaSubdivisionName,ActivityStartDate,ActivityStartTime/Time,ActivityStartTime/TimeZoneCode,ActivityEndDate,...,MethodDescriptionText,LaboratoryName,AnalysisStartDate,ResultLaboratoryCommentText,DetectionQuantitationLimitTypeName,DetectionQuantitationLimitMeasure/MeasureValue,DetectionQuantitationLimitMeasure/MeasureUnitCode,PreparationStartDate,ProviderName,source_file
0,USGS-AK,USGS Alaska Water Science Center,nwisak.01.98600200,Sample-Routine,Water,Surface Water,1985-10-24,16:00:00,AKDT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS,results_batch_001.csv
1,USGS-AK,USGS Alaska Water Science Center,nwisak.01.98600200,Sample-Routine,Water,Surface Water,1985-10-24,16:00:00,AKDT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS,results_batch_001.csv
2,USGS-AK,USGS Alaska Water Science Center,nwisak.01.98600200,Sample-Routine,Water,Surface Water,1985-10-24,16:00:00,AKDT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS,results_batch_001.csv
3,USGS-AK,USGS Alaska Water Science Center,nwisak.01.98600200,Sample-Routine,Water,Surface Water,1985-10-24,16:00:00,AKDT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS,results_batch_001.csv
4,USGS-AK,USGS Alaska Water Science Center,nwisak.01.98600200,Sample-Routine,Water,Surface Water,1985-10-24,16:00:00,AKDT,NaN,...,"NWIS User's Manual, QW System, Section 3.6.7",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS,results_batch_001.csv


In [5]:
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 6608 entries, 0 to 6607
Data columns (total 64 columns):
 #   Column                                             Non-Null Count  Dtype 
---  ------                                             --------------  ----- 
 0   OrganizationIdentifier                             6608 non-null   object
 1   OrganizationFormalName                             6608 non-null   object
 2   ActivityIdentifier                                 6608 non-null   object
 3   ActivityTypeCode                                   6608 non-null   object
 4   ActivityMediaName                                  6608 non-null   object
 5   ActivityMediaSubdivisionName                       6608 non-null   object
 6   ActivityStartDate                                  6608 non-null   object
 7   ActivityStartTime/Time                             4876 non-null   object
 8   ActivityStartTime/TimeZoneCode                     4876 non-null   object
 9   ActivityEndDate               

In [6]:
# Inspect the different measurement values:
print(results["CharacteristicName"].value_counts().to_string())

CharacteristicName
Stream flow, instantaneous                                      1354
Suspended Sediment Concentration (SSC)                           856
Suspended Sediment Discharge                                     629
Bedload sediment particle size                                   529
Total dissolved solids                                           245
Nitrate                                                          194
Specific conductance                                             183
Temperature, water                                               158
pH                                                               130
Alkalinity                                                       127
Height, gage                                                     116
Acidity, (H+)                                                    109
Chloride                                                         109
Sulfate                                                          108
Carbon dioxide 

The results above show that there many different characteristics are measured across sites, and many of these measurements are not relevant to this study. Target columns will be filtered below:

In [7]:
targets = [
    "Turbidity",
    "Temperature, water",
    "Specific conductance",
    "Oxygen",
    "Dissolved oxygen (DO)",
    "pH",
    "Nitrate",
    "Ammonia and ammonium",
    "Orthophosphate",
    "Phosphorus",
    "Total dissolved solids",
]

for target in targets:
    subset = results[results["CharacteristicName"] == target]

    print(
        f"{target:30s} "
        f"Results: {len(subset):5,} | "
        f"Stations: {subset['MonitoringLocationIdentifier'].nunique():3,} | "
        f"Dates: {subset['ActivityStartDate'].nunique():4,}"
    )

Turbidity                      Results:     0 | Stations:   0 | Dates:    0
Temperature, water             Results:   158 | Stations:  16 | Dates:   86
Specific conductance           Results:   183 | Stations:  12 | Dates:  102
Oxygen                         Results:     6 | Stations:   2 | Dates:    3
Dissolved oxygen (DO)          Results:     0 | Stations:   0 | Dates:    0
pH                             Results:   130 | Stations:  12 | Dates:   75
Nitrate                        Results:   194 | Stations:  10 | Dates:   69
Ammonia and ammonium           Results:    38 | Stations:   6 | Dates:    5
Orthophosphate                 Results:    20 | Stations:   6 | Dates:    5
Phosphorus                     Results:    20 | Stations:   6 | Dates:    6
Total dissolved solids         Results:   245 | Stations:   7 | Dates:   66


In [8]:
# Examine sampling date range of pilot:
results["ActivityStartDate"] = pd.to_datetime(
    results["ActivityStartDate"],
    errors="coerce",
)

print("Earliest measurement:", results["ActivityStartDate"].min())
print("Latest measurement:", results["ActivityStartDate"].max())

Earliest measurement: 1948-10-18 00:00:00
Latest measurement: 2008-06-25 00:00:00


## Now filter measurements within +/- 3 years of the dam removal

In [22]:
# Compare downloaded station IDs with the current pilot selection.
selected_stations = pd.read_csv(
    RESULTS_DIR / "pilot_station_selection.csv",
    dtype={"MonitoringLocationIdentifier": "string"},
)

selected_ids = set(
    selected_stations["MonitoringLocationIdentifier"]
    .dropna()
    .str.strip()
)

downloaded_ids = set(
    results["MonitoringLocationIdentifier"]
    .dropna()
    .str.strip()
)

print("Selected stations:", len(selected_ids))
print("Downloaded stations:", len(downloaded_ids))
print("Selected but not downloaded:", len(selected_ids - downloaded_ids))
print("Downloaded but not selected:", len(downloaded_ids - selected_ids))

Selected stations: 26
Downloaded stations: 21
Selected but not downloaded: 5
Downloaded but not selected: 0


In [23]:
# Download relevant data 
LINKS_FILE = PROJECT_ROOT / "data/processed/dam_station_links_19020401.csv"
DAMS_FILE = PROJECT_ROOT / "data/raw/dams/ARDamRemovalList_Figshare_Mar2026.csv"

links = pd.read_csv(
    LINKS_FILE,
    dtype={"MonitoringLocationIdentifier": "string"},
)

dams = pd.read_csv(DAMS_FILE)

display(links.head())
display(dams.head())
print("Link columns:", links.columns.tolist())
print("Dam columns:", dams.columns.tolist())

,huc8,dam_id,dam_name,station_id,dam_reach_id,year_removed,river,station_reach_id,relationship,distance_to_dam_km,network_distance_km,distance_method,dam_snap_distance_m,station_snap_distance_m,dam_snap_fraction,station_snap_fraction
0,19020401,AK-010,Lower Eklutna River Dam,USGS-15272970,75000100004714,2017,Eklutna,75000100002520,unconnected,NaN,NaN,NaN,227.764842,17.116171,0.521747,0.302300
1,19020401,AK-010,Lower Eklutna River Dam,USGS-15273020,75000100004714,2017,Eklutna,75000100000099,unconnected,NaN,NaN,NaN,227.764842,7.968422,0.521747,0.977877
2,19020401,AK-010,Lower Eklutna River Dam,USGS-15273030,75000100004714,2017,Eklutna,75000100001891,unconnected,NaN,NaN,NaN,227.764842,13.624231,0.521747,0.515974
3,19020401,AK-010,Lower Eklutna River Dam,USGS-15273040,75000100004714,2017,Eklutna,75000100000059,unconnected,NaN,NaN,NaN,227.764842,27.595394,0.521747,0.138479
4,19020401,AK-010,Lower Eklutna River Dam,USGS-15273045,75000100004714,2017,Eklutna,75000100000059,unconnected,NaN,NaN,NaN,227.764842,3.819037,0.521747,0.464142


,AR_ID,NABI_ID,NID_ID,Dam_Name,Year_Removed,Latitude,Longitude,City_County,River,HUC8,...,Dam_Length_ft,Year_Built,Owner,Original_Use,Type_Material,Reason_for_removal,Miles_Restored_US,Miles_Restored_DS,River_Miles_Reported,Last_Updated
0,AK-001,AK91779,NaN,Lake Bettinger Complex Dam,NaN,57.80333,-152.395,Kodiak,NaN,Kodiak-Afognak Islands,...,251,1959,City of Kodiak,Water Supply,Timber,Delapidated,NaN,NaN,NaN,2/10/2026
1,AK-002,AK91780,NaN,Bettinger Lower Reservoir Dam B,NaN,57.80167,-152.395,Kodiak,NaN,Kodiak-Afognak Islands,...,245,1959,City of Kodiak,Water Supply,Timber,Safety,NaN,NaN,NaN,2/10/2026
2,AK-003,AK91781,NaN,Memory Estates Dam #1,NaN,62.61833,-149.427,Matanuska-Susitna,NaN,Talkeetna River,...,379,1973,NaN,Recreation,Earthen,NaN,NaN,NaN,NaN,2/10/2026
3,AK-004,AK91782,NaN,Memory Estates Dam #2,NaN,62.62667,-149.380,Matanuska-Susitna,NaN,Talkeetna River,...,444,1973,NaN,Recreation,Earthen,NaN,NaN,NaN,NaN,2/10/2026
4,AK-005,NaN,NaN,Switzer One Dam,1988,NaN,NaN,"Tongass National Forest, Juneau",Switzer Creek,Lynn Canal,...,NaN,NaN,NaN,NaN,NaN,Safety,NaN,NaN,NaN,11/10/2017


Link columns: ['huc8', 'dam_id', 'dam_name', 'station_id', 'dam_reach_id', 'year_removed', 'river', 'station_reach_id', 'relationship', 'distance_to_dam_km', 'network_distance_km', 'distance_method', 'dam_snap_distance_m', 'station_snap_distance_m', 'dam_snap_fraction', 'station_snap_fraction']
Dam columns: ['AR_ID', 'NABI_ID', 'NID_ID', 'Dam_Name', 'Year_Removed', 'Latitude', 'Longitude', 'City_County', 'River', 'HUC8', 'State', 'Dam_Height_ft', 'Dam_Length_ft', 'Year_Built', 'Owner', 'Original_Use', 'Type_Material', 'Reason_for_removal', 'Miles_Restored_US', 'Miles_Restored_DS', 'River_Miles_Reported', 'Last_Updated']


In [24]:
# Compare station identifiers before the join.
print("Unique downloaded station IDs:",
      results["MonitoringLocationIdentifier"].nunique())

print("Unique linked station IDs:",
      dam_station_dates["MonitoringLocationIdentifier"].nunique())

print("\nSample downloaded IDs:")
print(results["MonitoringLocationIdentifier"].dropna().astype(str).head(10).tolist())

print("\nSample linked IDs:")
print(dam_station_dates["MonitoringLocationIdentifier"].dropna().astype(str).head(10).tolist())

# Check overlap after normalizing whitespace and types.
result_ids = set(
    results["MonitoringLocationIdentifier"]
    .dropna().astype(str).str.strip()
)
linked_ids = set(
    dam_station_dates["MonitoringLocationIdentifier"]
    .dropna().astype(str).str.strip()
)

print("\nMatching station IDs:", len(result_ids & linked_ids))
print("Downloaded IDs not linked:", len(result_ids - linked_ids))
print("Linked IDs not downloaded:", len(linked_ids - result_ids))

Unique downloaded station IDs: 21
Unique linked station IDs: 391

Sample downloaded IDs:
['USGS-15277600', 'USGS-15277600', 'USGS-15277600', 'USGS-15277600', 'USGS-15277600', 'USGS-15277600', 'USGS-15277600', 'USGS-15277600', 'USGS-15277600', 'USGS-15277600']

Sample linked IDs:
['USGS-15272970', 'USGS-15273020', 'USGS-15273030', 'USGS-15273040', 'USGS-15273045', 'USGS-15273050', 'USGS-15273090', 'USGS-15273095', 'USGS-15273097', 'USGS-15273099']

Matching station IDs: 21
Downloaded IDs not linked: 0
Linked IDs not downloaded: 370


In [25]:
# Check how many downloaded measurements match dam-linked stations.
print("Downloaded measurement rows:", len(results))
print("Rows after station-dam join:", len(dam_results))

print("\nMeasurement date range:")
print(results["ActivityStartDate"].min(), "to",
      results["ActivityStartDate"].max())

print("\nRemoval years represented:")
print(dam_station_dates["year_removed"].value_counts().sort_index())

print("\nValid measurement dates:", results["ActivityStartDate"].notna().sum())
print("Rows with valid removal dates:", dam_results["removal_date"].notna().sum())

print("\nRows inside the window before filtering:")
in_window = dam_results["ActivityStartDate"].between(
    dam_results["window_start"],
    dam_results["window_end"],
    inclusive="both",
)
print(in_window.sum())

Downloaded measurement rows: 6608
Rows after station-dam join: 6608

Measurement date range:
1948-10-18 00:00:00 to 2008-06-25 00:00:00

Removal years represented:
year_removed
2017    391
Name: count, dtype: int64

Valid measurement dates: 6608
Rows with valid removal dates: 6608

Rows inside the window before filtering:
0


In [26]:
# Summarize measurement coverage by calendar year.
year_counts = (
    results.assign(
        measurement_year=results["ActivityStartDate"].dt.year
    )
    .groupby("measurement_year")
    .size()
    .rename("measurement_count")
    .reset_index()
)

display(year_counts.tail(20))

,measurement_year,measurement_count
0,1948,34
1,1949,426
2,1950,529
3,1951,714
4,1952,349
5,1953,46
6,1981,76
7,1983,63
8,1985,309
9,1986,398


In [21]:
# Create +/-3 year subset
dam_station_dates = (
    links[
        ["dam_id", "station_id", "year_removed"]
    ]
    .rename(columns={"station_id": "MonitoringLocationIdentifier"})
    .drop_duplicates()
    .copy()
)

dam_station_dates["year_removed"] = pd.to_numeric(
    dam_station_dates["year_removed"],
    errors="coerce",
)

dam_station_dates = dam_station_dates.dropna(
    subset=["year_removed"]
).copy()

dam_station_dates["year_removed"] = (
    dam_station_dates["year_removed"].astype(int)
)

# Approximate removal date using January 1 of the recorded removal year.
dam_station_dates["removal_date"] = pd.to_datetime(
    dam_station_dates["year_removed"].astype(str) + "-01-01",
    errors="coerce",
)

# Parse measurement dates.
results["ActivityStartDate"] = pd.to_datetime(
    results["ActivityStartDate"],
    errors="coerce",
)

# Join measurements to their linked dam(s).
dam_results = results.merge(
    dam_station_dates,
    on="MonitoringLocationIdentifier",
    how="inner",
    validate="many_to_many",
)

# Filter to the inclusive +/-3 calendar-year window.
dam_results["window_start"] = (
    dam_results["removal_date"] - pd.DateOffset(years=3)
)
dam_results["window_end"] = (
    dam_results["removal_date"] + pd.DateOffset(years=3)
)

pilot_results = dam_results.loc[
    dam_results["ActivityStartDate"].between(
        dam_results["window_start"],
        dam_results["window_end"],
        inclusive="both",
    )
].copy()

print(f"All downloaded results: {len(results):,}")
print(f"Dam-linked measurement records: {len(dam_results):,}")
print(f"Records within +/-3 years: {len(pilot_results):,}")
print(
    "Unique stations in window:",
    pilot_results["MonitoringLocationIdentifier"].nunique(),
)
print("Unique dams represented:", pilot_results["dam_id"].nunique())

display(pilot_results.head(10))

All downloaded results: 6,608
Dam-linked measurement records: 6,608
Records within +/-3 years: 0
Unique stations in window: 0
Unique dams represented: 0


,OrganizationIdentifier,OrganizationFormalName,ActivityIdentifier,ActivityTypeCode,ActivityMediaName,ActivityMediaSubdivisionName,ActivityStartDate,ActivityStartTime/Time,ActivityStartTime/TimeZoneCode,ActivityEndDate,...,DetectionQuantitationLimitMeasure/MeasureValue,DetectionQuantitationLimitMeasure/MeasureUnitCode,PreparationStartDate,ProviderName,source_file,dam_id,year_removed,removal_date,window_start,window_end


In [27]:
targets = [
    "Turbidity",
    "Temperature, water",
    "Specific conductance",
    "Oxygen",
    "Dissolved oxygen (DO)",
    "pH",
    "Nitrate",
    "Ammonia and ammonium",
    "Orthophosphate",
    "Phosphorus",
    "Total dissolved solids",
]

coverage = (
    pilot_results.groupby("CharacteristicName")
    .agg(
        results=("ActivityStartDate", "size"),
        stations=("MonitoringLocationIdentifier", "nunique"),
        dams=("dam_id", "nunique"),
        first_measurement=("ActivityStartDate", "min"),
        last_measurement=("ActivityStartDate", "max"),
    )
    .sort_values("results", ascending=False)
)

display(coverage)

for target in targets:
    subset = pilot_results[
        pilot_results["CharacteristicName"] == target
    ]

    print(
        f"{target:30s} "
        f"Results: {len(subset):6,} | "
        f"Stations: {subset['MonitoringLocationIdentifier'].nunique():4,} | "
        f"Dams: {subset['dam_id'].nunique():3,}"
    )

,results,stations,dams,first_measurement,last_measurement
CharacteristicName,,,,,


Turbidity                      Results:      0 | Stations:    0 | Dams:   0
Temperature, water             Results:      0 | Stations:    0 | Dams:   0
Specific conductance           Results:      0 | Stations:    0 | Dams:   0
Oxygen                         Results:      0 | Stations:    0 | Dams:   0
Dissolved oxygen (DO)          Results:      0 | Stations:    0 | Dams:   0
pH                             Results:      0 | Stations:    0 | Dams:   0
Nitrate                        Results:      0 | Stations:    0 | Dams:   0
Ammonia and ammonium           Results:      0 | Stations:    0 | Dams:   0
Orthophosphate                 Results:      0 | Stations:    0 | Dams:   0
Phosphorus                     Results:      0 | Stations:    0 | Dams:   0
Total dissolved solids         Results:      0 | Stations:    0 | Dams:   0


The above notebook cells show that there are no measurements within a three year window of the dam removal. In fact, there are no measurements since 2008 and the dam was removed in 2017. So, this example dam removal does not have enough data to be used in the final study.

In [28]:
# Inspect which dams have measurements in the analysis window.
display(
    pilot_results.groupby(["dam_id", "removal_date"])
    .agg(
        measurements=("ActivityStartDate", "size"),
        stations=("MonitoringLocationIdentifier", "nunique"),
        first_measurement=("ActivityStartDate", "min"),
        last_measurement=("ActivityStartDate", "max"),
    )
    .reset_index()
    .sort_values("measurements", ascending=False)
)

# Choose a dam ID from the table above.
selected_dam_id = pilot_results["dam_id"].dropna().iloc[0]

dam_measurements = (
    pilot_results[pilot_results["dam_id"] == selected_dam_id]
    .sort_values(
        ["MonitoringLocationIdentifier", "ActivityStartDate"]
    )
)

display(dam_measurements)

,dam_id,removal_date,measurements,stations,first_measurement,last_measurement


IndexError: single positional indexer is out-of-bounds